# 01 - Nạp và khảo sát dữ liệu thô

Đọc 7 file CSV trong `data/raw/` bằng Spark với schema tường minh, kiểm tra sơ bộ, ghi ra Parquet (`data/parquet/`) và tạo bộ dữ liệu mẫu 10 cửa hàng (`data/sample/`) để phát triển nhanh ở các bước sau.

In [ ]:
import sys
import time
from pathlib import Path

THU_MUC_HIEN_TAI = Path.cwd()
GOC = THU_MUC_HIEN_TAI.parent if THU_MUC_HIEN_TAI.name == "notebooks" else THU_MUC_HIEN_TAI
sys.path.append(str(GOC / "src"))

from cau_hinh import DATA_RAW, DATA_PARQUET, DATA_SAMPLE, FILE_DU_LIEU, tao_thu_muc
from spark_session import tao_spark

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType, DateType, BooleanType
)

tao_thu_muc()
spark = tao_spark("01_NapDuLieu")
spark

## Khai báo schema tường minh

Không dùng `inferSchema` vì tốn thời gian quét 2 lần và có thể suy luận sai kiểu (vd. `dcoilwtico` có dòng rỗng).

In [ ]:
luoc_do = {
    "train.csv": StructType([
        StructField("id", IntegerType(), False),
        StructField("date", DateType(), False),
        StructField("store_nbr", IntegerType(), False),
        StructField("family", StringType(), False),
        StructField("sales", DoubleType(), False),
        StructField("onpromotion", IntegerType(), False),
    ]),
    "test.csv": StructType([
        StructField("id", IntegerType(), False),
        StructField("date", DateType(), False),
        StructField("store_nbr", IntegerType(), False),
        StructField("family", StringType(), False),
        StructField("onpromotion", IntegerType(), False),
    ]),
    "stores.csv": StructType([
        StructField("store_nbr", IntegerType(), False),
        StructField("city", StringType(), False),
        StructField("state", StringType(), False),
        StructField("type", StringType(), False),
        StructField("cluster", IntegerType(), False),
    ]),
    "oil.csv": StructType([
        StructField("date", DateType(), False),
        StructField("dcoilwtico", DoubleType(), True),
    ]),
    "holidays_events.csv": StructType([
        StructField("date", DateType(), False),
        StructField("type", StringType(), False),
        StructField("locale", StringType(), False),
        StructField("locale_name", StringType(), False),
        StructField("description", StringType(), False),
        StructField("transferred", BooleanType(), False),
    ]),
    "transactions.csv": StructType([
        StructField("date", DateType(), False),
        StructField("store_nbr", IntegerType(), False),
        StructField("transactions", IntegerType(), False),
    ]),
    "sample_submission.csv": StructType([
        StructField("id", IntegerType(), False),
        StructField("sales", DoubleType(), False),
    ]),
}

assert set(luoc_do.keys()) == set(FILE_DU_LIEU), "Danh sach schema khong khop FILE_DU_LIEU trong cau_hinh.py"

## Đọc dữ liệu và kiểm tra sơ bộ

Với mỗi file: đếm số dòng và đếm số dòng vi phạm cột bắt buộc (không được null) để phát hiện sớm dữ liệu lỗi/schema sai.

In [ ]:
khung_du_lieu = {}
t0 = time.time()
for ten_file, schema in luoc_do.items():
    duong_dan = DATA_RAW / ten_file
    df = spark.read.csv(str(duong_dan), header=True, schema=schema)
    khung_du_lieu[ten_file] = df
    so_dong = df.count()
    cot_bat_buoc = [f.name for f in schema.fields if not f.nullable]
    so_loi = df.filter(" OR ".join(f"{c} IS NULL" for c in cot_bat_buoc)).count() if cot_bat_buoc else 0
    print(f"{ten_file:<24} so_dong={so_dong:<10} loi_schema(cot_bat_buoc_bi_null)={so_loi}")

print("Thoi gian doc 7 file (s):", round(time.time() - t0, 2))

In [ ]:
khung_du_lieu["train.csv"].show(5)
khung_du_lieu["stores.csv"].show(5)
khung_du_lieu["oil.csv"].show(5)

## Kiểm tra nghiệp vụ nhanh

Khoảng thời gian, số cửa hàng, số nhóm hàng, số giá trị thiếu của giá dầu — để đối chiếu với mô tả bộ dữ liệu Kaggle.

In [ ]:
df_train = khung_du_lieu["train.csv"]
df_test = khung_du_lieu["test.csv"]
df_oil = khung_du_lieu["oil.csv"]

print("Khoang ngay train:", df_train.select(F.min("date"), F.max("date")).first())
print("Khoang ngay test:", df_test.select(F.min("date"), F.max("date")).first())
print("So store_nbr (train):", df_train.select("store_nbr").distinct().count())
print("So family (train):", df_train.select("family").distinct().count())
print("So gia tri null trong dcoilwtico:", df_oil.filter(F.col("dcoilwtico").isNull()).count())

## Ghi Parquet (toàn bộ) và bộ dữ liệu mẫu

Bộ mẫu chỉ giữ lại 10 cửa hàng đầu tiên (cho `train`/`test`/`transactions`/`stores`), còn `oil` và `holidays_events` không gắn với cửa hàng nên giữ nguyên toàn bộ. Mục đích: lặp lại nhanh ở các notebook EDA/feature engineering trước khi chạy full data.

In [ ]:
t0 = time.time()
for ten_file, df in khung_du_lieu.items():
    ten = ten_file.replace(".csv", "")
    df.write.mode("overwrite").parquet(str(DATA_PARQUET / ten))
print("Thoi gian ghi parquet toan bo (s):", round(time.time() - t0, 2))

SO_STORE_MAU = 10
for ten_file in ["train.csv", "test.csv", "transactions.csv", "stores.csv"]:
    ten = ten_file.replace(".csv", "")
    khung_du_lieu[ten_file].filter(F.col("store_nbr") <= SO_STORE_MAU) \
        .write.mode("overwrite").parquet(str(DATA_SAMPLE / ten))
for ten_file in ["oil.csv", "holidays_events.csv"]:
    ten = ten_file.replace(".csv", "")
    khung_du_lieu[ten_file].write.mode("overwrite").parquet(str(DATA_SAMPLE / ten))

print(f"Da tao du lieu mau cho {SO_STORE_MAU} cua hang.")

## Kiểm tra đọc lại

In [ ]:
kt = spark.read.parquet(str(DATA_PARQUET / "train"))
print("Doc lai parquet train, so dong:", kt.count())
assert kt.count() == df_train.count()

kt_mau = spark.read.parquet(str(DATA_SAMPLE / "train"))
print("Doc lai parquet sample train, so dong:", kt_mau.count())
assert kt_mau.select("store_nbr").distinct().count() == SO_STORE_MAU

In [ ]:
spark.stop()